In [0]:
# =====================================================
# Load Item Details - Localized Aspects
# =====================================================

localized_aspects_path = (
    "gs://market-intelligence-raw/ebay/"
    "item_details__localized_aspects/"
)

localized_aspects_df = (
    spark.read
    .option("multiLine", "false")
    .json(localized_aspects_path + "*.jsonl.gz")
)

print("Total rows:", localized_aspects_df.count())
print("Total columns:", len(localized_aspects_df.columns))

localized_aspects_df.printSchema()

In [0]:
localized_aspects_df.display(10)

In [0]:
# =====================================================
# Child cardinality per parent
# =====================================================

from pyspark.sql import functions as F

aspect_counts = (
    localized_aspects_df
    .groupBy("_dlt_parent_id")
    .agg(
        F.count("*").alias("aspect_count")
    )
)

display(
    aspect_counts
    .agg(
        F.count("*").alias("parent_count"),
        F.avg("aspect_count").alias("avg_aspects"),
        F.expr("percentile_approx(aspect_count, 0.25)").alias("p25"),
        F.expr("percentile_approx(aspect_count, 0.50)").alias("median"),
        F.expr("percentile_approx(aspect_count, 0.75)").alias("p75"),
        F.min("aspect_count").alias("min_aspects"),
        F.max("aspect_count").alias("max_aspects")
    )
)

### Localized Aspects Cardinality

The localized product-aspects data was profiled to understand how many aspects are associated with each listing and how much variation exists across listings.

| Metric | Result |
|---|---:|
| Parent listings with aspects | 57,891 |
| Average aspects per listing | 12.25 |
| P25 | 6 |
| Median | 11 |
| P75 | 17 |
| Minimum | 1 |
| Maximum | 62 |

There are **57,891 parent listings with localized aspects**, with an average of **12.25 aspects per listing**.

The median listing contains **11 aspects**, while the 25th percentile contains 6 and the 75th percentile contains 17. This indicates that the number of available aspects varies considerably between listings.

The observed range extends from **1 aspect to 62 aspects per listing**. This suggests that some products have relatively limited attribute information, while others contain much richer localized attribute coverage.

The difference between the mean (12.25) and median (11) is relatively small, suggesting that the distribution is not dominated by an extreme number of highly populated listings, although the maximum of 62 shows that richer records do exist.

These localized aspects provide a potentially useful source of product attributes for downstream enrichment. However, the aspects should be treated as a separate semi-structured attribute layer rather than assuming that every aspect is present for every listing.

In [0]:
# =====================================================
# Validate localized-aspects → Item Details relationship
# =====================================================
item_details_path = (
    "gs://market-intelligence-raw/ebay/item_details/"
)

item_details_df = (
    spark.read
    .option("multiLine", "false")
    .json(item_details_path + "*.jsonl.gz")
)

# Parents represented in localized aspects
aspect_parent_ids = (
    localized_aspects_df
    .select(F.col("_dlt_parent_id").alias("parent_dlt_id"))
    .distinct()
)

# Current Item Details parent IDs
parent_ids = (
    item_details_df
    .select(F.col("_dlt_id").alias("parent_dlt_id"))
    .distinct()
)

matched_parents = (
    aspect_parent_ids
    .join(parent_ids, on="parent_dlt_id", how="inner")
    .count()
)

unmatched_parents = (
    aspect_parent_ids
    .join(parent_ids, on="parent_dlt_id", how="left_anti")
    .count()
)

print("Localized-aspect parent IDs:", aspect_parent_ids.count())
print("Matched to current Item Details:", matched_parents)
print("Not found in current Item Details:", unmatched_parents)

### Localized Aspects → Item Details Relationship

The relationship between the localized-aspects dataset and the current Item Details dataset was validated using the parent listing identifier.

| Validation | Result |
|---|---:|
| Localized-aspect parent IDs | 57,891 |
| Matched to current Item Details | 57,891 |
| Not found in current Item Details | 0 |

All **57,891 localized-aspect parent IDs** were successfully matched to the current Item Details dataset.

There are **0 orphaned parent IDs**, meaning every localized-aspect record currently has a corresponding parent listing in Item Details.

This confirms that the localized-aspects data can be joined to the current Item Details dataset without losing any of the observed parent-aspect relationships.

The relationship can therefore be treated as:

```text
Item Details
    │
    │ 1
    │
    └──────────< Localized Aspects
                   many
```

A single listing can contain multiple localized aspects, as established by the earlier cardinality analysis, while every localized-aspect parent ID maps back to an existing Item Details record.

For downstream processing, the Item Details table can therefore serve as the parent listing table, with localized aspects maintained as a child/semi-structured attribute dataset.


In [0]:
# =====================================================
# Localized aspect attribute profile
# =====================================================

aspect_name_profile = (
    localized_aspects_df
    .groupBy("name")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("_dlt_parent_id").alias("listing_count")
    )
    .orderBy(F.desc("listing_count"))
)

print(
    "Distinct attribute names:",
    aspect_name_profile.count()
)

display(
    aspect_name_profile.limit(50)
)

### Localized Aspect Attribute Findings

The localized-aspects table contains a wide range of product attributes. The most frequently observed attributes include:

| Attribute | Listings |
|---|---:|
| `Brand` | 42,016 |
| `Type` | 34,042 |
| `Country of Origin` | 25,135 |
| `Color` | 25,101 |
| `Material` | 20,019 |
| `Features` | 18,418 |
| `UPC` | 13,986 |
| `Model` | 10,456 |
| `MPN` | 10,382 |
| `Style` | 9,239 |
| `Size` | 7,655 |
| `Format` | 7,602 |
| `Genre` | 6,644 |
| `Item Length` | 5,826 |
| `Language` | 5,762 |

The most frequently occurring attributes are general product characteristics such as `Brand`, `Type`, `Country of Origin`, `Color`, and `Material`.

However, the attribute vocabulary becomes increasingly diverse further down the distribution. Examples include:


- `Placement on Vehicle`
- `Publisher`
- `ISBN`
- `Artist`
- `Character`
- `Compatible Model`
- `Platform`
- `Book Title`
- `Franchise`
- `Movie/TV Title`
- `Compatible Brand`

This variation provides strong evidence that the available attributes are category-dependent rather than a fixed set of columns shared across all listings.

For example, an automotive listing may contain Placement on Vehicle, while a book listing may contain `ISBN`, `Publisher`, or `Book Title`. A video-game listing may instead contain `Platform` or `Compatible Model`.

Therefore, the localized-aspects data should not be modeled as a conventional fixed-schema dimension where every attribute is expected to exist for every listing.

The current structure is better understood as a dynamic attribute-value layer, where:
```text
Listing
   │
   ├── Brand → ...
   ├── Color → ...
   ├── Material → ...
   ├── UPC → ...
   ├── Platform → ...
   └── Category-specific attributes → ...
```
The category-dependent nature of these attributes is important for the downstream data model. The localized-aspects dataset should remain flexible and preserve the original attribute names and values rather than forcing all possible attributes into a wide fixed schema.

For the Silver layer, the localized aspects can therefore be retained as a normalized/semi-structured attribute representation and selectively promoted into analytical columns only when a particular attribute is relevant to the business use case.